# Predicting next-month arrears
Protocol: `protocol.yaml`. Temporal holdout from `holdout.csv`, walk-forward CV (5 splits) on training rows only, selection by mean CV ROC AUC, holdout scored once.

**Feature decision:** `final_notice_sent` is recorded in the billing cycle *after* the billing month, i.e. after/around the outcome, so it is not known at prediction time and is excluded (it would leak the target). `meter_ref` and `billing_month` are excluded by the protocol.

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='42accdc3f8c6ae3514896d73516d98c54e4a27c5422d4fe88e5c4fe1b00a14f7'
assert hashlib.sha256(open('holdout.csv','rb').read()).hexdigest()=='903bb064736772428ae27c90c6c03ac7f9c28c7eb54a50c70b7601b731258ba6'
df=pd.read_csv('data.csv'); df['billing_month']=pd.to_datetime(df['billing_month'])
hold_idx=pd.read_csv('holdout.csv')['row'].to_numpy()
is_hold=df.index.isin(hold_idx)
train=df[~is_hold]; hold=df[is_hold]
print(len(train),len(hold),train.billing_month.max(),hold.billing_month.min())
assert train.billing_month.max()<hold.billing_month.min()

In [ ]:
TARGET='arrears_next_month'
CAT=['tariff']; NUM=['occupants','kwh','bill_eur','late_payments_12m']
FEATURES=CAT+NUM
def prep(scale):
    return ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),CAT),
                              ('n',StandardScaler() if scale else 'passthrough',NUM)])
# all candidates named before any scoring
candidates={
 'baseline_prior': Pipeline([('p',prep(False)),('m',DummyClassifier(strategy='prior'))]),
 'logreg_C1': Pipeline([('p',prep(True)),('m',LogisticRegression(C=1.0,max_iter=1000))]),
 'logreg_C0.05': Pipeline([('p',prep(True)),('m',LogisticRegression(C=0.05,max_iter=1000))]),
 'rf_d6': Pipeline([('p',prep(False)),('m',RandomForestClassifier(n_estimators=300,max_depth=6,min_samples_leaf=20,random_state=0,n_jobs=-1))]),
 'hgb_small': Pipeline([('p',prep(False)),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0))]),
}
print(list(candidates))

In [ ]:
# walk-forward CV by month inside training rows (expanding window)
months=np.sort(train.billing_month.unique()); K=5
first=len(months)//(K+1)
chunks=np.array_split(months[first:],K)
folds=[]
for i,ch in enumerate(chunks):
    tr=train.index[train.billing_month<ch[0]].to_numpy()
    va=train.index[train.billing_month.isin(ch)].to_numpy()
    folds.append((tr,va)); print(i,len(tr),len(va),pd.Timestamp(ch[0]).date(),pd.Timestamp(ch[-1]).date())
cv_scores={}
for n,mdl in candidates.items():
    s=[]
    for tr,va in folds:
        m=clone(mdl).fit(df.loc[tr,FEATURES],df.loc[tr,TARGET])
        s.append(float(roc_auc_score(df.loc[va,TARGET],m.predict_proba(df.loc[va,FEATURES])[:,1])))
    cv_scores[n]=s
cv=pd.DataFrame({n:{'mean':np.mean(s),'std':np.std(s)} for n,s in cv_scores.items()}).T
cv

In [ ]:
chosen=max(cv_scores,key=lambda n:np.mean(cv_scores[n])); print('chosen',chosen)
final=clone(candidates[chosen]).fit(train[FEATURES],train[TARGET])
holdout_auc=float(roc_auc_score(hold[TARGET],final.predict_proba(hold[FEATURES])[:,1]))
print('holdout roc_auc',holdout_auc)

In [ ]:
ev={'features':FEATURES,'candidates':list(candidates),
 'cv_folds':[[tr.tolist(),va.tolist()] for tr,va in folds],
 'cv_scores':cv_scores,'chosen':chosen,
 'holdout_rows':[int(i) for i in hold.index],
 'holdout_scores':{chosen:holdout_auc}}
json.dump(ev,open('evidence.json','w'))

## Conclusion
See the CV table and holdout ROC AUC above. The holdout score was computed once, for the CV-chosen candidate only. `final_notice_sent` was deliberately left out as post-outcome information; including it would inflate performance without being usable at prediction time.